# 02 — LLM Provider Comparison

Compares LLM providers for agent mobility decisions:
- **Ollama + llama3.1:8b** (local)
- **vLLM + llama3.1:8b** (local server)
- **Google Gemini-2.0-Flash-Lite** (free API via OpenAI-compatible endpoint)

**Run order:** Cell 1 → Cell 2 → Cell 3 → any provider cell(s) → Summary → Plot  
Each provider cell is **independent** — run only the ones you have access to.

**Metrics**: Latency · Humanistic accuracy · JSON compliance · Token usage

In [ ]:
import asyncio
import json
import time
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from openai import AsyncOpenAI

sns.set_theme(style="whitegrid", palette="muted")

with open("data/sample_agents.json") as f:
    AGENTS = json.load(f)

PROVIDERS = {
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "api_key":  "ollama",
        "model":    "Qwen3.5-9B-GGUF:Q4_K_M",
    },
    "vllm": {
        "base_url": "http://localhost:8000/v1",
        "api_key":  "vllm",
        "model":    "Qwen/Qwen2.5-3B-Instruct-AWQ",
    },
    "gemini-flash": {
        "base_url": "https://generativelanguage.googleapis.com/v1beta/openai/",
        "api_key":  os.getenv("GEMINI_API_KEY", ""),
        "model":    "gemini-2.0-flash-lite",
    },
}

In [24]:
MOBILITY_PROMPT_TEMPLATE = """You are a {archetype} in Barcelona Eixample.
Current needs: hunger={hunger:.1f}, energy={energy:.1f}, social={social:.1f}
Candidate destinations (name, type, distance_m):
{candidates}

Choose the most appropriate destination for your archetype and needs.
Reply with JSON only: {{"destination": "<name>", "type": "<type>", "reasoning": "<brief>"}}"""

SAMPLE_CANDIDATES = [
    ("Bar Montana",                 "cafe",        45),
    ("Farmacia Roca",               "pharmacy",    80),
    ("Parc de la Ciutadella",       "park",       150),
    ("Mercado de la Sagrada Familia","supermarket",200),
    ("Sagrada Familia",             "attraction", 300),
]

def build_prompt(agent):
    cands = "\n".join(f"  - {n} ({t}, {d}m)" for n, t, d in SAMPLE_CANDIDATES)
    return MOBILITY_PROMPT_TEMPLATE.format(
        archetype=agent["archetype"],
        hunger=agent["needs"]["hunger"],
        energy=agent["needs"]["energy"],
        social=agent["needs"]["social"],
        candidates=cands,
    )

async def bench_provider(name, config, agents, runs_per_agent=3):
    """Benchmark one provider.  Returns rows with status='ok' or 'access_not_possible'."""
    kwargs = {"api_key": config["api_key"]}
    if config["base_url"]:
        kwargs["base_url"] = config["base_url"]

    client = AsyncOpenAI(**kwargs)

    # ── connectivity probe ──────────────────────────────────────────────────
    probe_prompt = "Reply with the single word: hello"
    try:
        await client.chat.completions.create(
            model=config["model"],
            messages=[{"role": "user", "content": probe_prompt}],
            max_tokens=10,
            timeout=10,
        )
    except Exception as probe_err:
        reason = str(probe_err)
        if "Connection" in reason or "connect" in reason.lower() or "refused" in reason.lower():
            reason = "service unreachable (not running locally)"
        elif "api_key" in reason.lower() or "authentication" in reason.lower() or "unauthorized" in reason.lower() or "invalid" in reason.lower():
            reason = "authentication failed (check API key)"
        print(f"  [{name}] ACCESS NOT POSSIBLE — {reason}")
        return [{"provider": name, "status": "access_not_possible", "error": reason,
                 "latency_ms": None, "json_ok": None, "accurate": None,
                 "in_tokens": None, "out_tokens": None}]

    # ── real benchmark ──────────────────────────────────────────────────────
    results = []
    for agent in agents[:5]:
        prompt = build_prompt(agent)
        for _ in range(runs_per_agent):
            t0 = time.perf_counter()
            try:
                resp = await client.chat.completions.create(
                    model=config["model"],
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=150,
                    temperature=0.7,
                )
                elapsed = (time.perf_counter() - t0) * 1000
                content = resp.choices[0].message.content or ""
                try:
                    parsed   = json.loads(content)
                    json_ok  = True
                    dest_type = parsed.get("type", "")
                except Exception:
                    json_ok   = False
                    dest_type = ""
                accurate  = dest_type in agent.get("expected_destinations", [])
                in_tok    = resp.usage.prompt_tokens     if resp.usage else 0
                out_tok   = resp.usage.completion_tokens if resp.usage else 0
                results.append({"provider": name, "status": "ok",
                                 "latency_ms": elapsed, "json_ok": json_ok,
                                 "accurate": accurate,
                                 "in_tokens": in_tok, "out_tokens": out_tok})
            except Exception as e:
                results.append({"provider": name, "status": "error",
                                 "error": str(e), "latency_ms": None,
                                 "json_ok": False, "accurate": False,
                                 "in_tokens": 0, "out_tokens": 0})
    return results

In [25]:
# ── Shared results store — run this before any provider cell ─────────────────
RESULTS = {}   # keyed by provider name; populated by each provider cell below
print("Results store initialised. Now run one or more provider cells below.")

Results store initialised. Now run one or more provider cells below.


### Provider 1 — Ollama (local)

In [26]:
# ── Ollama  (local · llama3.1:8b) ──────────────────────────────────────────────────────────
_name   = "ollama"
_config = PROVIDERS[_name]

print(f"Benchmarking {_name} ...")
_rows = await bench_provider(_name, _config, AGENTS)

RESULTS[_name] = _rows

_ok = [r for r in _rows if r["status"] == "ok"]
if _ok:
    _df = pd.DataFrame(_ok)
    print(f"  ✓ {len(_ok)} samples collected")
    print(_df[["latency_ms", "json_ok", "accurate"]].describe().round(2).to_string())
else:
    _err = _rows[0].get("error", "unknown") if _rows else "no response"
    print(f"  ✗ ACCESS NOT POSSIBLE — {_err}")

Benchmarking ollama ...
  [ollama] ACCESS NOT POSSIBLE — Error code: 404 - {'error': {'message': "model 'Qwen3.5-9B-GGUF:Q4_K_M' not found", 'type': 'api_error', 'param': None, 'code': None}}
  ✗ ACCESS NOT POSSIBLE — Error code: 404 - {'error': {'message': "model 'Qwen3.5-9B-GGUF:Q4_K_M' not found", 'type': 'api_error', 'param': None, 'code': None}}


### Provider 2 — vLLM (local)

In [ ]:
# ── vLLM    (local · llama3.1:8b) ──────────────────────────────────────────────────────────
_name   = "vllm"
_config = PROVIDERS[_name]

print(f"Benchmarking {_name} ...")
_rows = await bench_provider(_name, _config, AGENTS)

RESULTS[_name] = _rows

_ok = [r for r in _rows if r["status"] == "ok"]
if _ok:
    _df = pd.DataFrame(_ok)
    print(f"  ✓ {len(_ok)} samples collected")
    print(_df[["latency_ms", "json_ok", "accurate"]].describe().round(2).to_string())
else:
    _err = _rows[0].get("error", "unknown") if _rows else "no response"
    print(f"  ✗ ACCESS NOT POSSIBLE — {_err}")

### Provider 3 — Gemini-2.0-Flash-Lite (free API)

In [27]:
# ─ Gemini-2.0-Flash-Lite  (free API) ──────────────────────────────────────────────────────────
_name   = "gemini-flash"
_config = PROVIDERS[_name]

print(f"Benchmarking {_name} ...")
_rows = await bench_provider(_name, _config, AGENTS)

RESULTS[_name] = _rows

_ok = [r for r in _rows if r["status"] == "ok"]
if _ok:
    _df = pd.DataFrame(_ok)
    print(f"  ✓ {len(_ok)} samples collected")
    print(_df[["latency_ms", "json_ok", "accurate"]].describe().round(2).to_string())
else:
    _err = _rows[0].get("error", "unknown") if _rows else "no response"
    print(f"  ✗ ACCESS NOT POSSIBLE — {_err}")

Benchmarking gemini-flash ...
  [gemini-flash] ACCESS NOT POSSIBLE — authentication failed (check API key)
  ✗ ACCESS NOT POSSIBLE — authentication failed (check API key)


### Summary & Export  *(run after all desired provider cells)*

In [ ]:
import pandas as pd

if not RESULTS:
    print("No provider results yet — run at least one provider cell above.")
else:
    all_rows = [r for rows in RESULTS.values() for r in rows]
    df = pd.DataFrame(all_rows)
    df.to_csv("results_02_llm_raw.csv", index=False)

    accessible = df[df["status"] == "ok"]
    blocked    = df[df["status"] == "access_not_possible"]["provider"].unique()

    if not accessible.empty:
        summary = (
            accessible
            .groupby("provider")
            .agg(
                samples      =("latency_ms", "count"),
                median_lat_ms=("latency_ms", "median"),
                json_ok_pct  =("json_ok",    "mean"),
                accuracy_pct =("accurate",   "mean"),
            )
            .assign(json_ok_pct  =lambda x: (x.json_ok_pct   * 100).round(1),
                    accuracy_pct =lambda x: (x.accuracy_pct  * 100).round(1),
                    median_lat_ms=lambda x:  x.median_lat_ms.round(0))
        )
        print("=== Benchmark Results ===")
        print(summary.to_string())

    if len(blocked):
        print("\n=== ACCESS NOT POSSIBLE ===")
        for p in blocked:
            err = df[df["provider"] == p]["error"].iloc[0]
            print(f"  {p}: {err}")

    if accessible.empty:
        print("No providers returned results.")

### Visualisation  *(run after Summary cell)*

In [ ]:
_all_rows  = [r for rows in RESULTS.values() for r in rows]
df         = pd.DataFrame(_all_rows)
accessible = df[df["status"] == "ok"]
blocked    = list(df[df["status"] == "access_not_possible"]["provider"].unique())

if accessible.empty:
    print("Nothing to plot — no accessible providers.")
else:
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Violin — latency
    sns.violinplot(data=accessible, x="provider", y="latency_ms",
                   ax=axes[0], inner="box")
    axes[0].set_title("Response Latency by Provider")
    axes[0].set_ylabel("Latency (ms)")
    axes[0].tick_params(axis="x", rotation=15)

    # Bar — humanistic accuracy
    acc = accessible.groupby("provider")["accurate"].mean() * 100
    acc.plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="white")
    axes[1].set_title("Humanistic Accuracy (%)")
    axes[1].set_ylabel("Archetype-Appropriate Decisions (%)")
    axes[1].set_ylim(0, 100)
    axes[1].tick_params(axis="x", rotation=15)

    # Scatter — latency vs accuracy
    smry = accessible.groupby("provider").agg(
        latency_ms=("latency_ms", "median"),
        accurate  =("accurate",   "mean"),
    ).reset_index()
    axes[2].scatter(smry["latency_ms"], smry["accurate"] * 100, s=120)
    for _, row in smry.iterrows():
        axes[2].annotate(row["provider"], (row["latency_ms"], row["accurate"] * 100),
                         textcoords="offset points", xytext=(5, 5))
    axes[2].set_title("Latency vs Accuracy Trade-off")
    axes[2].set_xlabel("Median Latency (ms)")
    axes[2].set_ylabel("Accuracy (%)")

    if blocked:
        fig.text(0.5, -0.04,
                 "ACCESS NOT POSSIBLE: " + ", ".join(blocked),
                 ha="center", fontsize=10, color="red")

    plt.tight_layout()
    plt.savefig("results_02_llm_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()